In [1]:
# ============================================================
# NOTEBOOK 07
# MAGNETIC FIELD CONTROL / CURRENT OPTIMIZATION
# ============================================================

import numpy as np
from pathlib import Path

# ============================================================
# LOAD MATRICES
# ============================================================

project_root = Path.cwd().parent

data_dir = project_root / "data"

# ------------------------------------------------------------
# LOAD FULL MATRICES
# ------------------------------------------------------------

Ax = np.load(data_dir / "Ax.npy")
Ay = np.load(data_dir / "Ay.npy")
Az = np.load(data_dir / "Az.npy")

A = np.load(data_dir / "A.npy")

print("Ax shape:", Ax.shape)
print("Ay shape:", Ay.shape)
print("Az shape:", Az.shape)
print("A shape :", A.shape)

Ax shape: (6, 7)
Ay shape: (6, 7)
Az shape: (6, 7)
A shape : (18, 7)


In [2]:
# ============================================================
# COIL LABELS
# ============================================================

coil_names = [

    'X1',
    'X2',
    'X3',

    'Y1',
    'Y2',

    'Z1',
    'Z2'
]

print(coil_names)

['X1', 'X2', 'X3', 'Y1', 'Y2', 'Z1', 'Z2']


In [3]:
# ============================================================
# BUILD REDUCED MATRIX
# ============================================================
#
# Unknown currents:
#
# X1
# X2
# X3
# Y_group
# Z_group
#
# ============================================================

A_red = np.column_stack([

    A[:,0],                  # X1
    A[:,1],                  # X2
    A[:,2],                  # X3

    A[:,3] + A[:,4],         # Y grouped

    A[:,5] + A[:,6]          # Z grouped
])

print("Reduced matrix shape:", A_red.shape)

Reduced matrix shape: (18, 5)


In [5]:
# ============================================================
# AMBIENT FIELD
# ============================================================
#
# Units: mG
#
# ============================================================

Bx_ambient = -283.38
By_ambient =  106.79
Bz_ambient = -381.21

In [11]:
# ============================================================
# BUILD TARGET VECTOR
# ============================================================
#
# We want:
#
# B_coils + B_ambient = 0
#
# Therefore:
#
# B_coils = -B_ambient
#
# ============================================================
#ambient field
B_ambient = np.array([
    -283.38,
    106.79,
    -381.21
])

target_single = -B_ambient

Bx_target = np.full(6, target_single[0])
By_target = np.full(6, target_single[1])
Bz_target = np.full(6, target_single[2])

B_target = np.concatenate([
    Bx_target,
    By_target,
    Bz_target
])
print(B_target.shape)
print(B_target)

(18,)
[ 283.38  283.38  283.38  283.38  283.38  283.38 -106.79 -106.79 -106.79
 -106.79 -106.79 -106.79  381.21  381.21  381.21  381.21  381.21  381.21]


In [12]:
# ============================================================
# SOLVE LEAST SQUARES
# ============================================================

I_opt, residuals, rank, s = np.linalg.lstsq(

    A_red,
    B_target,

    rcond=None
)

# ============================================================
# LABELS
# ============================================================

labels = [

    'X1',
    'X2',
    'X3',

    'Y_group',
    'Z_group'
]

# ============================================================
# PRINT RESULTS
# ============================================================

print("\nOptimized currents:\n")

for name, current in zip(labels, I_opt):

    print(f"{name}: {current:.2f} A")


Optimized currents:

X1: 60.99 A
X2: 33.36 A
X3: 68.45 A
Y_group: -23.20 A
Z_group: 71.84 A


In [13]:
# ============================================================
# NOTEBOOK 07
# ARBITRARY FIELD CONTROL
# ============================================================

import numpy as np
from pathlib import Path

# ============================================================
# LOAD MATRIX
# ============================================================

project_root = Path.cwd().parent

data_dir = project_root / "data"

A = np.load(data_dir / "A.npy")

print("A shape:", A.shape)

A shape: (18, 7)


In [14]:
# ============================================================
# BUILD REDUCED MATRIX
# ============================================================
#
# Unknown currents:
#
# X1
# X2
# X3
# Y_group
# Z_group
#
# ============================================================

A_red = np.column_stack([

    A[:,0],                  # X1
    A[:,1],                  # X2
    A[:,2],                  # X3

    A[:,3] + A[:,4],         # Y grouped

    A[:,5] + A[:,6]          # Z grouped
])

print("Reduced matrix shape:", A_red.shape)

Reduced matrix shape: (18, 5)


In [15]:
# ============================================================
# USER INPUT FIELD
# ============================================================
#
# Desired TOTAL field at PMTs
#
# Units: mG
#
# ============================================================

Bx_in = float(input("Desired Bx at PMTs (mG): "))
By_in = float(input("Desired By at PMTs (mG): "))
Bz_in = float(input("Desired Bz at PMTs (mG): "))

B_in = np.array([

    Bx_in,
    By_in,
    Bz_in
])

print("\nDesired total field:")
print(B_in)

Desired Bx at PMTs (mG):  0
Desired By at PMTs (mG):  0
Desired Bz at PMTs (mG):  0



Desired total field:
[0. 0. 0.]


In [16]:
# ============================================================
# AMBIENT FIELD
# ============================================================

B_ambient = np.array([

    -283.38,
     106.79,
    -381.21
])

print("\nAmbient field:")
print(B_ambient)


Ambient field:
[-283.38  106.79 -381.21]


In [17]:
# ============================================================
# REQUIRED COIL FIELD
# ============================================================
#
# B_coils =
# B_desired - B_ambient
#
# ============================================================

target_single = B_in - B_ambient

print("\nRequired coil-generated field:")
print(target_single)


Required coil-generated field:
[ 283.38 -106.79  381.21]


In [18]:
# ============================================================
# BUILD TARGET VECTOR
# ============================================================

Bx_target = np.full(

    6,
    target_single[0]
)

By_target = np.full(

    6,
    target_single[1]
)

Bz_target = np.full(

    6,
    target_single[2]
)

# ------------------------------------------------------------
# STACK INTO SINGLE VECTOR
# ------------------------------------------------------------

B_target = np.concatenate([

    Bx_target,
    By_target,
    Bz_target
])

print("\nB_target shape:", B_target.shape)


B_target shape: (18,)


In [19]:
# ============================================================
# SOLVE LEAST SQUARES
# ============================================================

I_opt, residuals, rank, s = np.linalg.lstsq(

    A_red,
    B_target,

    rcond=None
)

In [20]:
# ============================================================
# PRINT CURRENTS
# ============================================================

labels = [

    'X1',
    'X2',
    'X3',

    'Y_group',
    'Z_group'
]

print("\nOptimized currents:\n")

for name, current in zip(labels, I_opt):

    print(f"{name}: {current:.2f} A")


Optimized currents:

X1: 60.99 A
X2: 33.36 A
X3: 68.45 A
Y_group: -23.20 A
Z_group: 71.84 A


In [ ]:
# ============================================================
# VALIDATION
# ============================================================

B_coils = A_red @ I_opt

# ------------------------------------------------------------
# ADD AMBIENT FIELD
# ------------------------------------------------------------

Bx_total = B_coils[0:6]   + B_ambient[0]
By_total = B_coils[6:12]  + B_ambient[1]
Bz_total = B_coils[12:18] + B_ambient[2]

print("\nResulting fields at PMTs:\n")

for i in range(6):

    Bmag = np.sqrt(

        Bx_total[i]**2 +
        By_total[i]**2 +
        Bz_total[i]**2
    )

    print(f"\nPMT{i+1}")
    print("-"*30)

    print(f"Bx   = {Bx_total[i]:.2f} mG")
    print(f"By   = {By_total[i]:.2f} mG")
    print(f"Bz   = {Bz_total[i]:.2f} mG")

    print(f"|B|  = {Bmag:.2f} mG")